# 01 — Data Cleaning

**Capstone:** AI/ML-Enabled Transaction Workflow Optimization for Group Operations — Stage-1 Application Intake and Validation
**Author:** Percival Hurditt — Walsh College, QM640, May 2026

## Purpose
This notebook documents the cleaning pipeline applied to the BPIC12 and BPIC17 event logs and produces the processed datasets consumed by all downstream notebooks (02–06).

The cleaning steps mirror Table 3 of the Interim Report and produce:
- `data/processed/case_level_balanced_1000.csv`
- `data/processed/stage_month_panel.csv`
- `data/processed/activity_to_stage_mapping.csv`

## How this notebook is structured
If raw BPIC XES files are present in `data/raw/`, the notebook runs the full cleaning pipeline. Otherwise, it prints a message explaining how to obtain the raw data and confirms that the synthetic processed CSVs (generated by `scripts/generate_synthetic_data.py`) are in place.


In [1]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)
print('Project root :', ROOT)
print('Raw dir      :', RAW)
print('Processed dir:', PROC)

Project root : /home/claude/capstone_package
Raw dir      : /home/claude/capstone_package/data/raw
Processed dir: /home/claude/capstone_package/data/processed


## Step 1 — locate raw event logs

BPIC12 and BPIC17 are publicly available from the 4TU.ResearchData repository. The download script `data/raw/download_bpic_data.py` will fetch them on demand. If they are absent, this notebook switches to documentation mode.

In [2]:
bpic12 = list(RAW.glob('*BPI*2012*.xes*')) + list(RAW.glob('*bpic12*.xes*'))
bpic17 = list(RAW.glob('*BPI*2017*.xes*')) + list(RAW.glob('*bpic17*.xes*'))
RAW_AVAILABLE = bool(bpic12) and bool(bpic17)
print('BPIC12 raw file present:', RAW_AVAILABLE and bpic12[0].name)
print('BPIC17 raw file present:', RAW_AVAILABLE and bpic17[0].name)
print()
if not RAW_AVAILABLE:
    print('Raw XES files not found. To download them, run:')
    print('   python data/raw/download_bpic_data.py')
    print()
    print('Documentation mode: this notebook will print the cleaning')
    print('logic without executing it. The synthetic processed CSVs')
    print('produced by scripts/generate_synthetic_data.py are used by')
    print('all downstream notebooks (02-06) and are sufficient to run')
    print('the full analytical pipeline end-to-end.')

BPIC12 raw file present: False
BPIC17 raw file present: False

Raw XES files not found. To download them, run:
   python data/raw/download_bpic_data.py

Documentation mode: this notebook will print the cleaning
logic without executing it. The synthetic processed CSVs
produced by scripts/generate_synthetic_data.py are used by
all downstream notebooks (02-06) and are sufficient to run
the full analytical pipeline end-to-end.


## Step 2 — XES → tabular conversion (executed only if raw files present)

The conversion uses `pm4py`. If the package is not installed and raw files are not present, this cell is skipped without error.

In [3]:
def xes_to_dataframe(path):
    """Convert a BPIC XES log to a pandas DataFrame."""
    import pm4py
    log = pm4py.read_xes(str(path))
    df = pm4py.convert_to_dataframe(log)
    df = df.rename(columns={
        'case:concept:name': 'case_id',
        'concept:name': 'activity_name',
        'time:timestamp': 'timestamp',
        'lifecycle:transition': 'lifecycle_transition',
        'org:resource': 'resource',
        'case:AMOUNT_REQ': 'requested_amount',
    })
    return df

if RAW_AVAILABLE:
    print('Converting BPIC12...')
    df12 = xes_to_dataframe(bpic12[0])
    df12['era'] = 0
    print(f'  {len(df12):,} events, {df12.case_id.nunique():,} cases')
    print('Converting BPIC17...')
    df17 = xes_to_dataframe(bpic17[0])
    df17['era'] = 1
    print(f'  {len(df17):,} events, {df17.case_id.nunique():,} cases')
    events = pd.concat([df12, df17], ignore_index=True)
else:
    events = None
    print('Skipping conversion (raw files not present).')

Skipping conversion (raw files not present).


## Step 3 — cleaning (Table 3 of the Interim Report)

Each step is implemented as a function so the same logic can be reused by the author when running on real data.

In [4]:
def clean_events(events: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Apply the seven cleaning steps from Table 3.
    Returns (cleaned_events, audit_dict)."""
    audit = {}
    n0 = len(events)

    # 1. truncated cases at log boundaries
    events['timestamp'] = pd.to_datetime(events['timestamp'], utc=True)
    case_first = events.groupby('case_id')['timestamp'].min()
    case_last  = events.groupby('case_id')['timestamp'].max()
    log_start  = case_first.min()
    log_end    = case_last.max()
    truncated  = case_first[case_first <= log_start + pd.Timedelta('7D')].index.union(
                 case_last[case_last  >= log_end   - pd.Timedelta('7D')].index)
    events = events[~events['case_id'].isin(truncated)].copy()
    audit['truncated_cases_excluded'] = len(truncated)

    # 2. timezone normalize (already utc)

    # 3. negative durations: detect at case+activity level
    events = events.sort_values(['case_id', 'timestamp'])
    events['delta'] = events.groupby('case_id')['timestamp'].diff()
    bad = events.loc[events['delta'] < pd.Timedelta(0), 'case_id'].unique()
    events = events[~events['case_id'].isin(bad)]
    audit['negative_duration_cases_excluded'] = len(bad)
    events = events.drop(columns=['delta'])

    # 4. duplicate events
    before = len(events)
    events = events.drop_duplicates(
        subset=['case_id','activity_name','timestamp','resource'])
    audit['duplicate_events_removed'] = before - len(events)

    # 5. requested_amount missing — median imputation within product band
    if events['requested_amount'].isna().any():
        med = events.groupby('era')['requested_amount'].transform('median')
        events['requested_amount'] = events['requested_amount'].fillna(med)
        audit['requested_amount_imputed'] = True

    # 6. outlier processing times — winsorize at P99.5 (applied later at
    #    the case duration level, not here)

    # 7. resource sparsity — pool low-frequency resources
    counts = events['resource'].value_counts()
    rare = counts[counts < 50].index
    events.loc[events['resource'].isin(rare), 'resource'] = 'other_resource'
    audit['rare_resources_pooled'] = int(len(rare))

    audit['events_in']  = n0
    audit['events_out'] = len(events)
    return events, audit

if events is not None:
    events_clean, audit = clean_events(events)
    for k, v in audit.items():
        print(f'  {k:35s} {v}')
else:
    print('Cleaning logic defined; not executed without raw data.')

Cleaning logic defined; not executed without raw data.


## Step 4 — derive case-level features and the balanced 1,000-case sample

The risk_flag target is constructed as:
`risk_flag = 1 if (any A_Incomplete event) OR (any stage-level SLA breach) else 0`.

In [5]:
STAGE1_ACTIVITIES = [
    'A_Create Application', 'A_Concept', 'A_Accepted', 'A_Complete',
    'A_Validating', 'A_Incomplete', 'W_Validate application', 'W_Handle leads',
]

def derive_case_level(events_clean: pd.DataFrame) -> pd.DataFrame:
    """Compute case-level features used by RQ1 and RQ2."""
    s1 = events_clean[events_clean['activity_name'].isin(STAGE1_ACTIVITIES)]
    feat = s1.groupby('case_id').agg(
        stage1_processing_time = ('timestamp',
            lambda x: (x.max() - x.min()).total_seconds() / 3600),
        completeness_flag = ('activity_name',
            lambda x: int('A_Incomplete' not in set(x))),
        rework_indicator = ('activity_name',
            lambda x: int((x == 'A_Incomplete').sum())),
        exception_flag = ('activity_name',
            lambda x: int('A_Incomplete' in set(x))),
        requested_amount = ('requested_amount', 'first'),
        era = ('era', 'first'),
    ).reset_index()
    feat['stage1_processing_time'] = (
        feat['stage1_processing_time']
            .clip(upper=feat['stage1_processing_time'].quantile(0.995))
    )
    sla_threshold = feat['stage1_processing_time'].quantile(0.75)
    feat['sla_breach_flag'] = (
        feat['stage1_processing_time'] > sla_threshold).astype(int)
    feat['risk_flag'] = (
        (feat['exception_flag'] == 1) | (feat['sla_breach_flag'] == 1)
    ).astype(int)
    return feat

if events is not None:
    case_level = derive_case_level(events_clean)
    print('case-level rows:', len(case_level))
    print('positive rate:', case_level.risk_flag.mean().round(3))
    pos = case_level[case_level.risk_flag == 1].sample(500, random_state=42)
    neg = case_level[case_level.risk_flag == 0].sample(500, random_state=42)
    balanced = pd.concat([pos, neg]).sample(frac=1, random_state=42)
    balanced.to_csv(PROC / 'case_level_balanced_1000.csv', index=False)
    print('wrote', PROC / 'case_level_balanced_1000.csv')
else:
    print('Skipping (raw not present). Verifying synthetic CSV is in place...')
    assert (PROC / 'case_level_balanced_1000.csv').exists(), (
        'Run scripts/generate_synthetic_data.py first.')
    print('  OK:', (PROC / 'case_level_balanced_1000.csv'))

Skipping (raw not present). Verifying synthetic CSV is in place...
  OK: /home/claude/capstone_package/data/processed/case_level_balanced_1000.csv


## Step 5 — verify processed datasets

Confirm that all three downstream-consumed CSVs are present and structurally valid before handing off to notebooks 02–06.

In [6]:
for fname in ['data_dictionary.csv', 'case_level_balanced_1000.csv',
              'stage_month_panel.csv', 'activity_to_stage_mapping.csv']:
    fp = PROC / fname
    df = pd.read_csv(fp)
    print(f'{fname:40s} {len(df):5d} rows × {df.shape[1]:2d} cols')
    print('  cols:', list(df.columns))
    print()

data_dictionary.csv                         23 rows ×  7 cols
  cols: ['variable_name', 'description', 'data_type', 'source', 'example_value', 'business_meaning', 'use_in_model_or_analysis']

case_level_balanced_1000.csv              1000 rows × 18 cols
  cols: ['case_id', 'requested_amount', 'submission_channel', 'product_type', 'product_complexity_index', 'completeness_flag', 'exception_flag', 'rework_indicator', 'stage1_processing_time', 'sla_breach_flag', 'risk_flag', 'customer_segment', 'recent_exception_rate', 'resource_workload', 'era', 'month_index', 'approval_verification_outcome', 'final_transaction_outcome']

stage_month_panel.csv                       76 rows × 10 cols
  cols: ['stage_id', 'stage_name', 'month_index', 'era', 'transaction_volume', 'mean_processing_time', 'exception_rate', 'sla_breach_rate', 'first_time_right', 'reconciliation_breaks']

activity_to_stage_mapping.csv               20 rows ×  3 cols
  cols: ['activity_name', 'stage_id', 'stage_name']



## Output

All three processed datasets are now available under `data/processed/`. Notebooks 02–06 read from this directory and require no further preparation.
